### Set up VizDoom

In [ ]:
from vizdoom import *
import numpy as np
import random
import time

In [ ]:
# Demo with random actions
EPISODES = 5

game = DoomGame()
game.load_config('ViZDoom/scenarios/basic.cfg')
game.init()

actions = np.identity(3, dtype=np.uint8)

print('REWARDS')
print('--------------------')
for episode in range(EPISODES):
    game.new_episode()

    while not game.is_episode_finished():
        state = game.get_state()
        frame = state.screen_buffer
        info = state.game_variables
        reward = game.make_action(random.choice(actions))
        time.sleep(0.25) # Delay between frames
    
    print(f'Episode {episode + 1} = {game.get_total_reward()}') # Max score is 101
    time.sleep(2)

game.close()

---
### Convert to Gymnasium environment

In [ ]:
from gymnasium import Env
from gymnasium.spaces import Discrete, Box
import cv2

In [ ]:
class VizDoomGym(Env):
    def __init__(self, render=False):
        super().__init__()
        self.game = DoomGame()
        self.game.load_config('ViZDoom/scenarios/basic.cfg')
        self.game.set_window_visible(render)
        self.game.init()

        self.observation_space = Box(low=0, high=255, shape=(100, 160, 1), dtype=np.uint8)
        self.action_space = Discrete(3)

    def step(self, action):
        actions = np.identity(3, dtype=np.uint8)
        reward = self.game.make_action(actions[action], 4)  # Skip 4 frames
        state = self.game.get_state()
        terminated = self.game.is_episode_finished()
        truncated = False

        if state:
            frame = state.screen_buffer
            frame = self.grayscale(frame)
            info = {'info': state.game_variables[0]}
        else:
            frame = np.zeros(self.observation_space.shape, dtype=np.uint8)
            info = {'info': 0}

        return frame, reward, terminated, truncated, info

    def render(self):
        pass

    def grayscale(self, observation):
        frame = cv2.cvtColor(np.moveaxis(observation, 0, -1), cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, (160, 100), interpolation=cv2.INTER_CUBIC)
        frame = np.reshape(frame, (100, 160, 1))

        return frame

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.game.new_episode()
        state = self.game.get_state()

        if state is None:
            frame = np.zeros(self.observation_space.shape, dtype=np.uint8)
        else:
            frame = self.grayscale(state.screen_buffer)

        return frame, {}

    def close(self):
        self.game.close()


---
### Set up Callback

In [ ]:
import os
from stable_baselines3.common.callbacks import BaseCallback

class TrainAndLoggingCallback(BaseCallback):
    def __init__(self, check_freq, save_path, verbose=1):
        super(TrainAndLoggingCallback, self).__init__(verbose)
        self.check_freq = check_freq
        self.save_path = save_path

    def _init_callback(self):
        if self.save_path:
            os.makedirs(self.save_path, exist_ok=True)

    def _on_step(self):
        if self.n_calls % self.check_freq == 0:
            model_path = os.path.join(self.save_path, f'best_model_{self.n_calls}')
            self.model.save(model_path)
        
        return True

In [ ]:
CHECKPOINT_DIR = "./train/basic"
LOG_DIR = "./log/basic"

In [ ]:
callback = TrainAndLoggingCallback(check_freq=10000, save_path=CHECKPOINT_DIR)

---
### Train the agent

In [ ]:
from stable_baselines3 import PPO
import torch

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

In [ ]:
env = VizDoomGym()

In [ ]:
model = PPO('CnnPolicy', env, tensorboard_log=LOG_DIR, verbose=1, learning_rate=1e-4, n_steps=4096, device=device)

In [ ]:
model.learn(total_timesteps=100000, callback=callback)

---
### Test the agent

In [ ]:
EPISODES = 5
model = PPO.load("./train/train_basic/best_model_190000")
env = VizDoomGym(render=True)

print('REWARDS')
print('--------------------')
for episode in range(5):
    obs, _ = env.reset()
    terminated = False
    truncated = False
    total_reward = 0

    while not (terminated or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        time.sleep(0.25)
        total_reward += reward

    print(f"Episode {episode + 1} = {total_reward}")
    time.sleep(2)

env.close()

In [ ]:
# Bulk testing
from stable_baselines3.common.evaluation import evaluate_policy

mean_reward, _ = evaluate_policy(model, env, n_eval_episodes=100)
print(f"Mean reward = {mean_reward}")